# Stage 1 ASR Evaluation

**Whisper Large-v3 vs Parakeet CTC 0.6B Vietnamese**

This notebook is the control and review layer for the Stage 1 evaluation. All metric computation remains in `src/evaluation.py` and `src/06_evaluate_stage1_models.py`.

The core10 regression gate has already passed. The next normal step is `development`, which evaluates the development queries against the full 50-video retrieval corpus while keeping holdout results closed.

## 1. Setup


### 1.1 Mount Drive and load the evaluation configuration

This section defines project paths, loads the frozen evaluation configuration, and checks that the required source files are present.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display


PROJECT_ROOT    = Path("/content/drive/MyDrive/aic26/asr_model_comparison")
CONFIG_PATH     = PROJECT_ROOT / "configs" / "stage1_evaluation.json"
RUNNER_PATH     = PROJECT_ROOT / "src" / "06_evaluate_stage1_models.py"
EVALUATION_PATH = PROJECT_ROOT / "src" / "evaluation.py"

assert PROJECT_ROOT.exists(), PROJECT_ROOT
assert CONFIG_PATH.exists(), CONFIG_PATH
assert RUNNER_PATH.exists(), RUNNER_PATH
assert EVALUATION_PATH.exists(), EVALUATION_PATH

config = json.loads(CONFIG_PATH.read_text(encoding = "utf-8"))

REFERENCE_MODEL = config["reference_model_id"]
CANDIDATE_MODEL = config["candidate_model_ids"][0]

pd.options.display.max_colwidth = 140
pd.options.display.float_format = "{:.4f}".format

print("Project root      :", PROJECT_ROOT)
print("Evaluation ID     :", config["evaluation_id"])
print("Evaluation frozen :", config["evaluation_frozen"])
print("Reference model   :", REFERENCE_MODEL)
print("Candidate model   :", CANDIDATE_MODEL)
print("Runner            :", RUNNER_PATH)

Project root      : /content/drive/MyDrive/aic26/asr_model_comparison
Evaluation ID     : stage1_asr_parakeet_vs_whisper_large_v3
Evaluation frozen : False
Reference model   : whisper_large_v3
Candidate model   : parakeet_ctc_0_6b_vietnamese
Runner            : /content/drive/MyDrive/aic26/asr_model_comparison/src/06_evaluate_stage1_models.py


### 1.2 Select the evaluation mode

Use the modes in this order:

- `development` – current step; development20 queries against the full 50-video corpus.
- `holdout` – run once only after the evaluator and configuration are frozen.
- `final` – generate the final complete report after holdout.

`regression` remains available for debugging, but it has already passed.

In [ ]:
RUN_MODE = "development"

assert RUN_MODE in {"regression", "development", "holdout", "final"}

if (RUN_MODE in {"holdout", "final"} and not config["evaluation_frozen"]) :
    raise RuntimeError(f"{RUN_MODE!r} is locked because evaluation_frozen is false.")

print("Selected mode:", RUN_MODE)

Selected mode: development


## 2. Run the evaluation

### 2.1 Execute the authoritative evaluation runner

The runner validates benchmark and model-output identity, rechecks the core10 regression gate, computes retrieval metrics and diagnostics, and writes reproducible reports. Progress is printed live by the updated source files.

In [ ]:
os.environ["ASR_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["STAGE1_EVALUATION_CONFIG"] = str(CONFIG_PATH)

started = time.time()

!python -u "$RUNNER_PATH" --mode "$RUN_MODE"

elapsed_min = (time.time() - started) / 60.0
print(f"\nEvaluation finished in {elapsed_min:.1f} minutes.")

STAGE 1 ASR EVALUATION
Evaluation: stage1_asr_parakeet_vs_whisper_large_v3
Mode:       development
Frozen:     False
Project:    /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Config:     /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/configs/stage1_evaluation.json
Reports:    /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/reports/stage1/full_evaluation/development

[1/8] Loading benchmark manifests...
      benchmark keys: core10, extension40
      done in 0.0s
[2/8] Validating benchmarks and model outputs...
      whisper_large_v3: loading K01_V009 (1/10)
      whisper_large_v3: loading K05_V018 (2/10)
      whisper_large_v3: loading K07_V030 (3/10)
      whisper_large_v3: loading K12_V001 (4/10)
      whisper_large_v3: loading K16_V010 (5/10)
      whisper_large_v3: loading K18_V002 (6/10)
      whisper_large_v3: loading L21_V

### 2.2 Load the generated reports

The notebook only reads the report files created by the runner. It does not recalculate the authoritative evaluation metrics.

In [ ]:
REPORT_BASE = PROJECT_ROOT / config["reports"]["root"]
REPORT_ROOT = REPORT_BASE if RUN_MODE == "final" else REPORT_BASE / RUN_MODE

assert REPORT_ROOT.exists(), REPORT_ROOT

def load_report_json(filename : str) :
    path = REPORT_ROOT / filename
    return json.loads(path.read_text(encoding = "utf-8")) if path.exists() else None


def load_report_csv(filename : str) -> pd.DataFrame :
    path = REPORT_ROOT / filename
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def show_columns(frame : pd.DataFrame, columns : list[str], maximum_rows : int = 50) -> None :
    available = [column for column in columns if column in frame.columns]
    display(frame[available].head(maximum_rows))


validation = load_report_json("validation_summary.json")
regression = load_report_json("regression_gate.json")
manifest = load_report_json("evaluation_manifest.json")
bundle = load_report_json("stage1_evaluation_bundle.json")

model_summary = load_report_csv("model_summary.csv")
retrieval_metrics = load_report_csv("retrieval_metrics.csv")
query_results = load_report_csv("query_results.csv")
query_comparison = load_report_csv("query_comparison.csv")
raw_vs_processed = load_report_csv("raw_vs_processed.csv")
window_diagnostics = load_report_csv("window_diagnostics.csv")
reference_agreement = load_report_csv("reference_agreement.csv")
duplicate_pairs = load_report_csv("duplicate_pairs.csv")
repeated_phrases = load_report_csv("repeated_phrases.csv")
operational = load_report_csv("operational_summary.csv")
postprocess_consistency = load_report_csv("postprocess_consistency.csv")
inspection_cases = load_report_json("inspection_cases.json") or []

print("Report directory:", REPORT_ROOT)
print("Generated files :", len([path for path in REPORT_ROOT.iterdir() if path.is_file()]))

Report directory: /content/drive/MyDrive/aic26/asr_model_comparison/reports/stage1/full_evaluation/development
Generated files : 17


## 3. Integrity and provenance

### 3.1 Validation and regression gate

A valid evaluation must pass both the data-integrity checks and the frozen core10 regression gate. The latter confirms that the updated evaluator still reproduces the original Whisper Large-v3 retrieval results.

In [ ]:
print("Integrity validation:", "PASS" if validation["passed"] else "FAIL")
print("Regression gate     :", "PASS" if regression["passed"] else "FAIL")

regression_checks = pd.DataFrame(regression["checks"])
show_columns(regression_checks, ["view", "metric", "expected", "actual", "difference", "passed"], maximum_rows = 20)

model_validation_rows = []

for item in validation["components"]["model_outputs"] :
    model_validation_rows.append({
        "model_id" : item["model_id"],
        "benchmark_key" : item.get("benchmark_key"),
        "passed" : item["passed"],
        "expected_windows" : item["expected_window_count"],
        "present_windows" : item["present_window_count"],
        "legacy_missing_stage_id" : item.get("legacy_missing_stage_id_count", 0),
        "legacy_missing_benchmark_hash" : item.get("legacy_missing_benchmark_hash_count", 0),
        "error_count" : len(item.get("errors", [])),
        "warning_count" : len(item.get("warnings", [])),
    })

display(pd.DataFrame(model_validation_rows))

Integrity validation: PASS
Regression gate     : PASS


,view,metric,expected,actual,difference,passed
0,processed,story_recall_at_1,0.8000,0.8000,0.0000,True
1,processed,story_recall_at_3,0.9000,0.9000,0.0000,True
2,processed,story_mrr,0.8583,0.8583,0.0000,True
3,processed,video_recall_at_1,0.8000,0.8000,0.0000,True
4,processed,video_mrr,0.8700,0.8700,0.0000,True
5,raw,story_recall_at_1,0.8000,0.8000,0.0000,True
6,raw,story_recall_at_3,0.9000,0.9000,0.0000,True
7,raw,story_mrr,0.8533,0.8533,0.0000,True
8,raw,video_recall_at_1,0.9000,0.9000,0.0000,True
9,raw,video_mrr,0.9167,0.9167,0.0000,True


,model_id,benchmark_key,passed,expected_windows,present_windows,legacy_missing_stage_id,legacy_missing_benchmark_hash,error_count,warning_count
0,whisper_large_v3,core10,True,219,219,219,219,0,2
1,parakeet_ctc_0_6b_vietnamese,core10,True,219,219,0,0,0,0
2,whisper_large_v3,extension40,True,775,775,0,0,0,0
3,parakeet_ctc_0_6b_vietnamese,extension40,True,775,775,0,0,0,0


### 3.2 Evaluation provenance and post-processing consistency

This confirms the exact evaluator, semantic model, corpus hashes, and post-processing consistency used for the current run.

In [ ]:
print("Evaluation version  :", manifest["evaluation_version"])
print("Postprocess version :", manifest["postprocess_version"])
print("Generated at        :", manifest["generated_at_utc"])
print("Config SHA-256      :", manifest["config_hash"])
print("Evaluation frozen   :", manifest["evaluation_frozen"])

print()
print("Semantic model:")
print(json.dumps(manifest["semantic_model"], ensure_ascii = False, indent = 2))

print()
print("Logical corpus hashes:")
print(json.dumps(manifest["logical_corpora"], ensure_ascii = False, indent = 2))

print()
print("Post-processing consistency:")
display(postprocess_consistency)

Evaluation version  : 2.1.0
Postprocess version : 1.1
Generated at        : 2026-08-09T18:05:40.284378+00:00
Config SHA-256      : bffb3dc8ce645ce8de7015fc5a66dfc78aceb9a40e359eb1bfff9cd405713f6a
Evaluation frozen   : False

Semantic model:
{
  "model_name": "intfloat/multilingual-e5-small",
  "revision": "614241f622f53c4eeff9890bdc4f31cfecc418b3",
  "query_prefix": "query: ",
  "passage_prefix": "passage: "
}

Logical corpus hashes:
{
  "core10": "ed05e4104c18ac37082c4275b8cd009478fe699ba719d1e944ab0eb7f7651ab3",
  "all50": "4c6d51cee36638b67bf99de3a343116a6393bcd75d9e7fbccb4d7de0fd623cf5"
}

Post-processing consistency:


,model_id,stage_id,present_window_count,postprocess_match_count,postprocess_mismatch_count,postprocess_match_rate,stored_postprocess_versions
0,parakeet_ctc_0_6b_vietnamese,stage1_10_video,219,219,0,1.0000,['1.1']
1,parakeet_ctc_0_6b_vietnamese,stage1_extension40,775,775,0,1.0000,['1.1']
2,whisper_large_v3,stage1_10_video,219,219,0,1.0000,['1.1']
3,whisper_large_v3,stage1_extension40,775,775,0,1.0000,['1.1']


## 4. Retrieval performance

### 4.1 Aggregate retrieval results

Story retrieval measures temporal localization inside the correct video. Video retrieval measures whether the correct video is ranked highly among the candidate videos.

Both raw and canonical processed ASR text are reported.

In [ ]:
overall_retrieval = retrieval_metrics[retrieval_metrics["breakdown_type"] == "overall"].copy()

query_sets = overall_retrieval["query_set"].dropna().unique().tolist()
FOCUS_QUERY_SET = "development20" if "development20" in query_sets else ("holdout20" if "holdout20" in query_sets else query_sets[-1])

focus_retrieval = overall_retrieval[overall_retrieval["query_set"] == FOCUS_QUERY_SET].copy()

print("Focus query set:", FOCUS_QUERY_SET)

show_columns(
    focus_retrieval,
    [
        "model_id", "view", "query_count",
        "story_recall_at_1", "story_recall_at_3", "story_recall_at_5", "story_recall_at_10", "story_mrr",
        "video_recall_at_1", "video_recall_at_3", "video_recall_at_5", "video_recall_at_10", "video_recall_at_20", "video_mrr",
        "story_score_margin_mean", "video_score_margin_mean",
    ],
    maximum_rows = 20,
)

Focus query set: development20


,model_id,view,query_count,story_recall_at_1,story_recall_at_3,story_recall_at_5,story_recall_at_10,story_mrr,video_recall_at_1,video_recall_at_3,video_recall_at_5,video_recall_at_10,video_recall_at_20,video_mrr,story_score_margin_mean,video_score_margin_mean
0,parakeet_ctc_0_6b_vietnamese,processed,20,0.7000,0.7500,0.8500,0.9500,0.7613,0.3500,0.7000,0.7500,0.7500,0.8500,0.5421,0.0319,0.0023
1,parakeet_ctc_0_6b_vietnamese,raw,20,0.7000,0.8000,0.9000,0.9500,0.7770,0.4000,0.7000,0.7500,0.8000,0.9000,0.5692,0.0319,0.0036
2,whisper_large_v3,processed,20,0.7000,0.8000,0.9000,0.9500,0.7845,0.6000,0.6500,0.7500,0.7500,0.9000,0.6610,0.0337,0.0072
3,whisper_large_v3,raw,20,0.7000,0.8000,0.9000,1.0000,0.7831,0.6000,0.7000,0.7500,0.7500,0.9000,0.6761,0.0314,0.0090


### 4.2 Per-query Whisper vs Parakeet comparison

This view shows where Parakeet performs better, ties, or performs worse than Whisper for the same query.

In [ ]:
focus_queries = query_comparison[query_comparison["query_set"] == FOCUS_QUERY_SET].copy()

story_relation = f"{CANDIDATE_MODEL}__processed__story_vs_reference"
video_relation = f"{CANDIDATE_MODEL}__processed__video_vs_reference"

show_columns(
    focus_queries,
    [
        "query_id", "query_text", "correct_video", "task_type", "difficulty", "query_category",
        f"{REFERENCE_MODEL}__processed__first_relevant_rank",
        f"{CANDIDATE_MODEL}__processed__first_relevant_rank",
        story_relation,
        f"{REFERENCE_MODEL}__processed__video_rank",
        f"{CANDIDATE_MODEL}__processed__video_rank",
        video_relation,
    ],
    maximum_rows = 100,
)

print()
print("Processed story comparison:")
display(focus_queries[story_relation].value_counts(dropna = False).rename("count").to_frame())

print("Processed video comparison:")
display(focus_queries[video_relation].value_counts(dropna = False).rename("count").to_frame())

,query_id,query_text,correct_video,task_type,difficulty,query_category,whisper_large_v3__processed__first_relevant_rank,parakeet_ctc_0_6b_vietnamese__processed__first_relevant_rank,parakeet_ctc_0_6b_vietnamese__processed__story_vs_reference,whisper_large_v3__processed__video_rank,parakeet_ctc_0_6b_vietnamese__processed__video_rank,parakeet_ctc_0_6b_vietnamese__processed__video_vs_reference
10,R1-13,"Đoạn video mô tả một người ngồi vệ sinh máy ảnh. Công đoạn này bắt đầu bằng việc tháo rời máy ảnh. Tiếp theo, chiếc ống kính đã được thá...",L30_V095,KIS,medium,other,4,4,tie,1,2,worse
11,R1-15,Đoạn video về một chương trình từ thiện của một câu lạc bộ tên là FANA. Trong đoạn video có thể thấy câu lạc bộ này đang đi trao quà tại...,L30_V072,QA,easy,person_organization_location,1,1,tie,1,1,tie
12,R1-17,Đoạn video trong buổi trao quà từ thiện diễn ra tại 1 bệnh viện trong dịp Xuân 2024. Trong cảnh có hai người đàn ông (mặc áo sơ mi hồng ...,L30_V092,KIS,medium,number_or_measurement,1,1,tie,2,3,worse
13,R1-2,Mẩu tin giới thiệu về đàn hổ tại một địa phương ở miền Nam vừa có thêm khoảng 3-6 con hổ con. Đây là một giống hổ quý hiếm,L21_V029,KIS,easy,number_or_measurement,1,1,tie,1,1,tie
14,R1-22,Đoạn video về một người phụ nữ dạy nấu ăn cho những người khác. Trong đoạn video có thể thấy một người đang cầm công thức món ăn với ngu...,L26_V178,QA,easy,number_or_measurement,4,6,worse,5,2,better
15,R1-23,Đoạn clip về 1 thị trấn ven biển thu hút du khách hiếu kỳ nhờ 1 loài động vật biển nguy hiểm. Loài động vật này nổi tiếng trong 1 bộ phi...,L22_V022,KIS,hard,number_or_measurement,1,1,tie,1,1,tie
16,R1-9,Đoạn clip là cảnh thu hoạch dứa ở miền Tây: một bà cụ ngồi bên giỏ dứa trò chuyện với cô gái mặc áo hồng quàng khăn rằn; xung quanh chất...,L27_V013,KIS,medium,person_organization_location,11,7,better,17,12,better
17,R2-1,"Nhiều người mặt áo cờ đỏ sao vàng đứng trước biểu tượng một con cua khổng lồ. Những người này đều đội nón, một số người quấn khăn rằn tr...",K03_V019,KIS,easy,other,10,4,better,12,21,worse
18,R2-11,"Cảnh quay một người phụ nữ lớn tuổi đang đọc sách, tay đeo một xâu hạt. Sau đó là cảnh quay người phụ này đang mở một quyển sách có bìa ...",L30_V014,KIS,medium,person_organization_location,1,1,tie,1,1,tie
19,R2-20,Đoạn video bắt đầu bằng cảnh một số người đang cào muối trên đồng. Cảnh tiếp theo là một đoàn người đang vẫy tay phía sau một bảng chữ. ...,K06_V010,KIS,medium,other,2,12,worse,32,18,better



Processed story comparison:


,count
parakeet_ctc_0_6b_vietnamese__processed__story_vs_reference,
tie,15
worse,3
better,2


Processed video comparison:


,count
parakeet_ctc_0_6b_vietnamese__processed__video_vs_reference,
worse,9
tie,7
better,4


### 4.3 Raw vs processed ASR text

This measures whether the frozen post-processing policy improves, leaves unchanged, or worsens retrieval rank for each model.

In [ ]:
focus_effects = raw_vs_processed[raw_vs_processed["query_set"] == FOCUS_QUERY_SET].copy()

story_effects = focus_effects.groupby(["model_id", "story_effect"]).size().rename("count").reset_index()
video_effects = focus_effects.groupby(["model_id", "video_effect"]).size().rename("count").reset_index()

print("Story retrieval:")
display(story_effects)

print("Video retrieval:")
display(video_effects)

Story retrieval:


,model_id,story_effect,count
0,parakeet_ctc_0_6b_vietnamese,improved,1
1,parakeet_ctc_0_6b_vietnamese,unchanged,15
2,parakeet_ctc_0_6b_vietnamese,worsened,4
3,whisper_large_v3,improved,1
4,whisper_large_v3,unchanged,17
5,whisper_large_v3,worsened,2


Video retrieval:


,model_id,video_effect,count
0,parakeet_ctc_0_6b_vietnamese,improved,3
1,parakeet_ctc_0_6b_vietnamese,unchanged,12
2,parakeet_ctc_0_6b_vietnamese,worsened,5
3,whisper_large_v3,improved,4
4,whisper_large_v3,unchanged,12
5,whisper_large_v3,worsened,4


## 5. Transcript agreement and reliability


### 5.1 Parakeet agreement with the Whisper pseudo-reference

Whisper Large-v3 is not ground truth. WER and CER here measure **agreement with Whisper**, so they should be interpreted together with retrieval performance and reliability diagnostics.

In [ ]:
agreement_summary = pd.DataFrame(bundle["reference_agreement"])
display(agreement_summary)

if (not reference_agreement.empty) :
    disagreement = reference_agreement.sort_values("symmetric_token_distance", ascending = False)

    print("Largest transcript disagreements:")
    show_columns(
        disagreement,
        [
            "video_id", "window_id", "reference_trusted", "reference_untrusted_reasons",
            "whisper_reference_wer", "whisper_reference_cer", "token_f1",
            "symmetric_token_distance", "candidate_reference_length_ratio", "semantic_similarity",
        ],
        maximum_rows = 15,
    )

,subset,window_count,whisper_reference_wer_mean,whisper_reference_wer_median,whisper_reference_cer_mean,whisper_reference_cer_median,token_f1_mean,character_similarity_mean,semantic_similarity_mean,length_ratio_mean
0,all_reference_windows,597,0.3657,0.2140,0.2933,0.1402,0.8073,0.7319,0.9612,1.1959
1,trusted_reference_windows,592,0.3328,0.2128,0.2652,0.1390,0.8137,0.7373,0.9621,1.1694


Largest transcript disagreements:


,video_id,window_id,reference_trusted,reference_untrusted_reasons,whisper_reference_wer,whisper_reference_cer,token_f1,symmetric_token_distance,candidate_reference_length_ratio,semantic_similarity
317,L26_V178,L26_V178_0007,False,"['reference_dominant_known_boilerplate', 'reference_exact_unrelated_duplicate']",1.0000,0.8727,0.0690,1.0000,0.9333,0.9003
364,L30_V014,L30_V014_0002,True,[],1.0000,0.9412,0.0000,1.0000,0.0909,0.8818
444,K12_V001,K12_V001_0000,False,"['reference_dominant_known_boilerplate', 'reference_exact_unrelated_duplicate']",7.2000,6.3727,0.0242,0.9908,7.2667,0.8289
140,K06_V010,K06_V010_0015,True,[],9.7778,10.0471,0.0405,0.9814,9.9630,0.8620
71,K03_V023,K03_V023_0023,False,"['reference_dominant_known_boilerplate', 'reference_exact_unrelated_duplicate']",6.7333,5.6909,0.0339,0.9806,6.8667,0.8396
592,L27_V014,L27_V014_0012,False,"['reference_dominant_known_boilerplate', 'reference_exact_unrelated_duplicate']",2.4333,1.8545,0.0762,0.9733,2.5000,0.8598
338,L29_V020,L29_V020_0000,False,"['reference_dominant_known_boilerplate', 'reference_exact_unrelated_duplicate']",3.9667,3.3273,0.0784,0.9675,4.1000,0.8249
443,K07_V030,K07_V030_0020,True,[],5.0000,4.9737,0.1067,0.9524,5.2500,0.8974
579,L22_V030,L22_V030_0024,True,[],6.2105,7.1786,0.1233,0.9291,6.6842,0.8391
24,K02_V005,K02_V005_0024,True,[],4.5455,5.4848,0.3056,0.8197,5.5455,0.9114


### 5.2 Repetition, duplicates, and repeated phrases

These are diagnostic signals only. Near duplicates and repeated phrases are not automatically rejected because news videos can legitimately share language.

In [ ]:
reliability_summary = model_summary[
    [
        "model_id", "invalid_timestamp_count", "processed_rejection_count",
        "exact_unrelated_duplicate_pairs", "repeated_phrase_count",
    ]
].copy()

display(reliability_summary)

if (not duplicate_pairs.empty) :
    exact_unrelated = duplicate_pairs[
        duplicate_pairs["exact_normalized"].fillna(False)
        & duplicate_pairs["relationship"].isin(["non_adjacent_same_video", "cross_video"])
    ].copy()

    print("Exact unrelated duplicate pairs:", len(exact_unrelated))
    show_columns(
        exact_unrelated,
        [
            "model_id", "relationship", "left_video_id", "left_window_id",
            "right_video_id", "right_window_id", "character_similarity", "token_f1", "semantic_similarity",
        ],
        maximum_rows = 20,
    )

if (not repeated_phrases.empty) :
    print("Most suspicious repeated phrases:")
    show_columns(
        repeated_phrases.sort_values("suspicious_score", ascending = False),
        [
            "model_id", "phrase", "token_length", "occurrence_count",
            "distinct_window_count", "distinct_video_count", "maximum_window_coverage", "suspicious_score",
        ],
        maximum_rows = 20,
    )

,model_id,invalid_timestamp_count,processed_rejection_count,exact_unrelated_duplicate_pairs,repeated_phrase_count
0,whisper_large_v3,46,5,4,500
1,parakeet_ctc_0_6b_vietnamese,0,0,0,500


Exact unrelated duplicate pairs: 4


,model_id,relationship,left_video_id,left_window_id,right_video_id,right_window_id,character_similarity,token_f1,semantic_similarity
0,whisper_large_v3,cross_video,K03_V023,K03_V023_0023,L26_V178,L26_V178_0007,1.0000,1.0000,1.0000
1,whisper_large_v3,cross_video,K12_V001,K12_V001_0000,L27_V014,L27_V014_0012,1.0000,1.0000,1.0000
2,whisper_large_v3,cross_video,L29_V020,L29_V020_0000,K12_V001,K12_V001_0000,1.0000,1.0000,1.0000
3,whisper_large_v3,cross_video,L29_V020,L29_V020_0000,L27_V014,L27_V014_0012,1.0000,1.0000,1.0000


Most suspicious repeated phrases:


,model_id,phrase,token_length,occurrence_count,distinct_window_count,distinct_video_count,maximum_window_coverage,suspicious_score
0,whisper_large_v3,để không bỏ lỡ những video hấp dẫn,8,68,65,28,0.5333,14.9333
6,whisper_large_v3,mì gõ để không bỏ lỡ những video,8,57,54,26,0.5333,13.8667
7,whisper_large_v3,subscribe cho kênh ghiền mì gõ để không,8,57,54,26,0.5333,13.8667
4,whisper_large_v3,hãy subscribe cho kênh ghiền mì gõ để,8,57,54,26,0.5333,13.8667
3,whisper_large_v3,gõ để không bỏ lỡ những video hấp,8,57,54,26,0.5333,13.8667
2,whisper_large_v3,ghiền mì gõ để không bỏ lỡ những,8,57,54,26,0.5333,13.8667
5,whisper_large_v3,kênh ghiền mì gõ để không bỏ lỡ,8,57,54,26,0.5333,13.8667
1,whisper_large_v3,cho kênh ghiền mì gõ để không bỏ,8,57,54,26,0.5333,13.8667
9,whisper_large_v3,để không bỏ lỡ những video hấp,7,68,65,28,0.4667,13.0667
8,whisper_large_v3,không bỏ lỡ những video hấp dẫn,7,68,65,28,0.4667,13.0667


## 6. Operational efficiency


### 6.1 Runtime, RTF, memory, and failures

These measurements show the practical cost of using each ASR model. They complement retrieval quality rather than replacing it.

In [ ]:
show_columns(
    operational,
    [
        "model_id", "stage_id", "expected_window_count", "successful_window_count", "failed_window_count", "empty_window_count",
        "total_inference_runtime_s", "aggregate_rtf", "median_window_rtf", "p90_window_rtf", "p95_window_rtf",
        "windows_per_hour", "peak_gpu_memory_bytes", "peak_reserved_memory_bytes",
    ],
    maximum_rows = 20,
)

if (set([REFERENCE_MODEL, CANDIDATE_MODEL]).issubset(set(model_summary["model_id"]))) :
    summary_by_model = model_summary.set_index("model_id")
    speedup = summary_by_model.loc[REFERENCE_MODEL, "aggregate_rtf"] / summary_by_model.loc[CANDIDATE_MODEL, "aggregate_rtf"]
    memory_ratio = summary_by_model.loc[REFERENCE_MODEL, "peak_gpu_memory_bytes"] / summary_by_model.loc[CANDIDATE_MODEL, "peak_gpu_memory_bytes"]

    print(f"Parakeet RTF speed advantage: {speedup:.1f}x")
    print(f"Whisper / Parakeet peak allocated GPU memory: {memory_ratio:.2f}x")

,model_id,stage_id,expected_window_count,successful_window_count,failed_window_count,empty_window_count,total_inference_runtime_s,aggregate_rtf,median_window_rtf,p90_window_rtf,p95_window_rtf,windows_per_hour,peak_gpu_memory_bytes,peak_reserved_memory_bytes
0,parakeet_ctc_0_6b_vietnamese,stage1_10_video,219,219,0,0,110.6929,0.0085,0.0084,0.0094,0.0097,7122.4101,4912930816.0000,5247074304.0000
1,parakeet_ctc_0_6b_vietnamese,stage1_extension40,378,378,0,0,200.2865,0.0090,0.0087,0.0099,0.0101,6794.2663,4921511936.0000,4940890112.0000
2,whisper_large_v3,stage1_10_video,219,219,0,0,4219.1019,0.3257,0.3355,0.3859,0.4037,186.8644,6744243712.0000,9875488768.0000
3,whisper_large_v3,stage1_extension40,378,378,0,0,6775.8233,0.3047,0.3105,0.3662,0.3945,200.8317,6745738752.0000,7000293376.0000


Parakeet RTF speed advantage: 35.4x
Whisper / Parakeet peak allocated GPU memory: 1.37x


## 7. Manual inspection


### 7.1 Automatically selected cases

The evaluator selects high-value cases such as model disagreements, retrieval wins/losses, repeated phrases, suspicious reference windows, and runtime outliers.

In [ ]:
inspection_frame = pd.DataFrame(inspection_cases)
display(inspection_frame)

,case_type,query_id,window_id,model_id,details
0,largest_transcript_disagreement,None,L26_V178_0007,parakeet_ctc_0_6b_vietnamese,"{'symmetric_token_distance': 1.0, 'whisper_reference_wer': 1.0}"
1,largest_length_disagreement,None,K06_V010_0015,parakeet_ctc_0_6b_vietnamese,{'candidate_reference_length_ratio': 9.962962962962964}
2,whisper_reference_suspicious,None,K03_V023_0023,whisper_large_v3,"{'reasons': ['reference_dominant_known_boilerplate', 'reference_exact_unrelated_duplicate']}"
3,parakeet_retrieval_win,R2-1,K03_V019_0000,parakeet_ctc_0_6b_vietnamese,"{'reference_story_rank': 10, 'candidate_story_rank': 4}"
4,whisper_retrieval_win,R2-20,K06_V010_0024,whisper_large_v3,"{'reference_story_rank': 2, 'candidate_story_rank': 12}"
5,both_retrieval_fail,R1-17,L30_V092_0001,None,"{'reference_video_rank': 2, 'candidate_video_rank': 3}"
6,raw_vs_processed_rank_change,R1-13,None,parakeet_ctc_0_6b_vietnamese,"{'story_effect': 'worsened', 'video_effect': 'worsened'}"
7,highest_whisper_repetition,None,K02_V005_0003,whisper_large_v3,{'intra_window_repetition_score': 0.5093457943925234}
8,highest_parakeet_repetition,None,K12_V001_0000,parakeet_ctc_0_6b_vietnamese,{'intra_window_repetition_score': 0.1574074074074074}
9,exact_cross_video_duplicate,None,K03_V023_0023,whisper_large_v3,"{'other_window_id': 'L26_V178_0007', 'other_video_id': 'L26_V178'}"


### 7.2 Inspect one case with local window context

For a selected window, the helper shows the previous, selected, and next ASR windows for both models. This is useful because neighboring benchmark windows overlap in time.

In [ ]:
core_manifest_path = PROJECT_ROOT / config["benchmarks"]["core10"]["path"]
core_manifest = json.loads(core_manifest_path.read_text(encoding = "utf-8"))
core_video_ids = set(core_manifest["selected_video_ids"])


def output_directory(model_id : str, video_id : str) -> Path :
    output_key = "core10_output" if video_id in core_video_ids else "extension40_output"
    path = Path(config["models"][model_id][output_key])
    return path if path.is_absolute() else PROJECT_ROOT / path


def load_video_result(model_id : str, video_id : str) -> dict :
    path = output_directory(model_id, video_id) / f"{video_id}.json"
    return json.loads(path.read_text(encoding = "utf-8"))


def inspect_window(window_id : str) -> None :
    video_id = window_id.rsplit("_", 1)[0]
    target_index = int(window_id.rsplit("_", 1)[1])
    rows = []

    for model_id in [REFERENCE_MODEL, CANDIDATE_MODEL] :
        payload = load_video_result(model_id, video_id)
        windows = {int(item["window_index"]) : item for item in payload["windows"]}

        for offset, relation in [(-1, "previous"), (0, "selected"), (1, "next")] :
            index = target_index + offset

            if (index not in windows) :
                continue

            item = windows[index]

            rows.append({
                "model_id" : model_id,
                "relation" : relation,
                "window_id" : item["window_id"],
                "start_s" : item["start_s"],
                "end_s" : item["end_s"],
                "status" : item.get("status"),
                "runtime_s" : item.get("runtime_s"),
                "warnings" : item.get("warning_reasons", []),
                "rejections" : item.get("rejection_reasons", []),
                "raw_text" : item.get("raw_text", ""),
                "retrieval_text" : item.get("retrieval_text", ""),
            })

    display(pd.DataFrame(rows))


def inspect_query(query_id : str) -> None :
    subset = query_results[(query_results["query_id"] == query_id) & (query_results["query_set"] == FOCUS_QUERY_SET)].copy()

    show_columns(
        subset,
        [
            "query_id", "query_text", "model_id", "view", "correct_video",
            "first_relevant_rank", "video_rank", "top_story_window_id",
            "top_story_score", "story_score_margin", "video_score_margin",
        ],
        maximum_rows = 20,
    )

In [ ]:
CASE_INDEX = 0

case = inspection_cases[CASE_INDEX]
print(json.dumps(case, ensure_ascii = False, indent = 2))

window_id = case.get("window_id")
query_id = case.get("query_id")

if (window_id) :
    print("\nWindow context")
    inspect_window(window_id)

if (query_id) :
    print("\nQuery result")
    inspect_query(query_id)

{
  "case_type": "largest_transcript_disagreement",
  "query_id": null,
  "window_id": "L26_V178_0007",
  "model_id": "parakeet_ctc_0_6b_vietnamese",
  "details": {
    "symmetric_token_distance": 1.0,
    "whisper_reference_wer": 1.0
  }
}

Window context


,model_id,relation,window_id,start_s,end_s,status,runtime_s,warnings,rejections,raw_text,retrieval_text
0,whisper_large_v3,previous,L26_V178_0006,270.0000,330.0000,ok,16.4318,[partial_known_boilerplate],[],"Đây, rất hấp dẫn Và để xem lại công thức của món bánh ít trần này Các bạn nhớ quét mạng QR đang hiển thị trên bàn hình nha Cô sẽ lần lượ...",đây rất hấp dẫn và để xem lại công thức của món bánh ít trần này các bạn nhớ quét mạng qr đang hiển thị trên bàn hình nha cô sẽ lần lượt...
1,whisper_large_v3,selected,L26_V178_0007,315.0000,330.7683,ok,1.5891,"[invalid_timestamps, known_boilerplate]",[dominant_known_boilerplate],Hãy subscribe cho kênh Ghiền Mì Gõ Để không bỏ lỡ những video hấp dẫn,
2,parakeet_ctc_0_6b_vietnamese,previous,L26_V178_0006,270.0000,330.0000,ok,0.4966,[],[],"Đây rấtt hấp dẫn và để xem lại công thức của món bánh ít trừng này, các bạn nhớ quét bã khuya r giờ đang hiển thị trên màn hình nhé. cô ...",đây rấtt hấp dẫn và để xem lại công thức của món bánh ít trừng này các bạn nhớ quét bã khuya r giờ đang hiển thị trên màn hình nhé cô sẽ...
3,parakeet_ctc_0_6b_vietnamese,selected,L26_V178_0007,315.0000,330.7683,ok,0.2158,[],[],Mónon mỗi ngày để tìm hiểu lại công thức của các món ăn ngon.,mónon mỗi ngày để tìm hiểu lại công thức của các món ăn ngon


## 8. Decision and next step


### 8.1 Compact evidence summary

Use retrieval quality as the primary selection evidence, then consider reliability, Whisper-reference agreement, and operational cost. The notebook does not automatically choose the model.

In [ ]:
decision_retrieval = focus_retrieval[focus_retrieval["view"] == "processed"].copy()

show_columns(
    decision_retrieval,
    [
        "model_id", "query_count",
        "story_recall_at_1", "story_recall_at_3", "story_recall_at_5", "story_mrr",
        "video_recall_at_1", "video_recall_at_3", "video_recall_at_5", "video_mrr",
        "story_score_margin_mean", "video_score_margin_mean",
    ],
    maximum_rows = 10,
)

print()
show_columns(
    model_summary,
    [
        "model_id", "aggregate_rtf", "peak_gpu_memory_bytes", "invalid_timestamp_count",
        "processed_rejection_count", "exact_unrelated_duplicate_pairs",
        "trusted_whisper_reference_wer_mean", "trusted_token_f1_mean", "trusted_semantic_similarity_mean",
    ],
    maximum_rows = 10,
)

,model_id,query_count,story_recall_at_1,story_recall_at_3,story_recall_at_5,story_mrr,video_recall_at_1,video_recall_at_3,video_recall_at_5,video_mrr,story_score_margin_mean,video_score_margin_mean
0,parakeet_ctc_0_6b_vietnamese,20,0.7000,0.7500,0.8500,0.7613,0.3500,0.7000,0.7500,0.5421,0.0319,0.0023
2,whisper_large_v3,20,0.7000,0.8000,0.9000,0.7845,0.6000,0.6500,0.7500,0.6610,0.0337,0.0072


,model_id,aggregate_rtf,peak_gpu_memory_bytes,invalid_timestamp_count,processed_rejection_count,exact_unrelated_duplicate_pairs,trusted_whisper_reference_wer_mean,trusted_token_f1_mean,trusted_semantic_similarity_mean
0,whisper_large_v3,0.3124,6745738752.0000,46,5,4,NaN,NaN,NaN
1,parakeet_ctc_0_6b_vietnamese,0.0088,4921511936.0000,0,0,0,0.3328,0.8137,0.9621


### 8.2 Record the Stage 1 decision

During `development`, leave the decision unsaved until manual review is complete. After that, freeze the evaluator/configuration deliberately, run `holdout` once, then run `final`.

The notebook does **not** automatically change `evaluation_frozen`; that remains an explicit experiment-control decision.

In [ ]:
STAGE1_DECISION = {
    "selected_strategy" : None,
    "primary_reason" : None,
    "retrieval_observation" : None,
    "reliability_observation" : None,
    "runtime_observation" : None,
    "limitations" : [
        "Whisper Large-v3 is a pseudo-reference, not ground-truth transcription.",
        "Temporal relevance uses silver supervision based on answer time ±60 seconds.",
        "Stage 1 measures ASR retrieval utility, not exact-frame retrieval quality.",
    ],
}

print(json.dumps(STAGE1_DECISION, ensure_ascii = False, indent = 2))

if (RUN_MODE == "final" and STAGE1_DECISION["selected_strategy"] is not None) :
    decision_path = REPORT_ROOT / "stage1_decision.json"
    decision_path.write_text(json.dumps(STAGE1_DECISION, ensure_ascii = False, indent = 2), encoding = "utf-8")
    print("Saved:", decision_path)
else :
    print("\nDecision not saved. Complete development review, freeze the evaluator, run holdout once, then run final.")

{
  "selected_strategy": null,
  "primary_reason": null,
  "retrieval_observation": null,
  "reliability_observation": null,
  "runtime_observation": null,
  "limitations": [
    "Whisper Large-v3 is a pseudo-reference, not ground-truth transcription.",
    "Temporal relevance uses silver supervision based on answer time ±60 seconds.",
    "Stage 1 measures ASR retrieval utility, not exact-frame retrieval quality."
  ]
}

Decision not saved. Complete development review, freeze the evaluator, run holdout once, then run final.
